# Costruzione del Dataframe e Data Understanding

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
with open('balance_data.json', 'r') as outfile:
    balance_data = json.load(outfile)

In [3]:
with open('enriched_institutions_final.json', 'r') as f:
    enriched_institutions_final = json.load(f)

In [4]:
with open('macroaree_by_institution_per_anno.json', 'r') as papers:
    macroaree_by_institution = json.load(papers)

Notato che i enriched_institutions_final e macroaree_by_institution hanno lunghezza diverse, investighiamo.

In [5]:
len(set([info.get("id") or info.get("openalex_id") for info in enriched_institutions_final.values()]))

ids_enriched = set(
    info.get("id") or info.get("openalex_id")
    for info in enriched_institutions_final.values()
)

ids_macro = set(macroaree_by_institution.keys())

print("enriched:", len(ids_enriched))
print("macro:", len(ids_macro))
print("missing:", ids_enriched - ids_macro)

enriched: 1287
macro: 1287
missing: set()


In [6]:
from collections import Counter

ids = [
    info.get("id") or info.get("openalex_id")
    for info in enriched_institutions_final.values()
]

counter = Counter(ids)

duplicati = {id_: count for id_, count in counter.items() if count > 1}

print(duplicati)

{'https://openalex.org/I4210132340': 2}


Avendo identificato l'id duplicato all'interno di enriched_institutions_final, procediamo a visualizzare come mai lo è. Si nota, poiché ho lavorato sui nomi (mai più), la stessa agenzia è presente con due nominativi diversi distinti solo da utilizzo diverso di maiuscole e minuscole. Procediamo con la rimozione di una delle due entrate.

In [7]:
for istituzione, info in enriched_institutions_final.items():
    id = info.get("id") or info.get("openalex_id")
    if id == 'https://openalex.org/I4210132340':
        print(istituzione)

Agenzia Regionale per la Protezione dell'ambiente ligure
Agenzia Regionale per la Protezione dell’Ambiente Ligure


In [8]:
# la eliminiamo
enriched_institutions_final.pop("Agenzia Regionale per la Protezione dell'ambiente ligure")

{'2020': {'tot_dipendenti_per_anno': 17,
  'studiosi_usciti': 6,
  'studiosi_entrati': 6},
 '2021': {'tot_dipendenti_per_anno': 13,
  'studiosi_usciti': 3,
  'studiosi_entrati': 2},
 '2022': {'tot_dipendenti_per_anno': 13,
  'studiosi_usciti': 4,
  'studiosi_entrati': 3},
 '2023': {'tot_dipendenti_per_anno': 12,
  'studiosi_usciti': 4,
  'studiosi_entrati': 3},
 '2024': {'tot_dipendenti_per_anno': 11,
  'studiosi_usciti': 3,
  'studiosi_entrati': 3},
 'id': 'https://openalex.org/I4210132340',
 'informazioni_geografiche': {'city': 'Genoa',
  'geonames_city_id': '3176219',
  'region': None,
  'country_code': 'IT',
  'country': 'Italy',
  'latitude': 44.41151428222656,
  'longitude': 8.884272575378418,
  'regione': 'Liguria',
  'macroarea': 'Nord'},
 'counts_by_year': [{'year': 2026,
   'works_count': 1,
   'oa_works_count': 1,
   'cited_by_count': 0},
  {'year': 2025,
   'works_count': 88,
   'oa_works_count': 82,
   'cited_by_count': 32},
  {'year': 2024,
   'works_count': 130,
   'oa_w

##### Per le istituzioni senza id
Queste linee di codice sono state usate per creare il file che conteneva le istituzioni originariamente senza id (chissà per quale motivo) che si trovavano in enriched_institutions. Il file enriched_institutions_senza_id si basa sul dizionario prodotto nell'ultima delle prossime tre celle.
Questo è stato arricchito ed unito ad institutions_no_id, così che adesso si può lavorare solo su enriched_institutions_final.

In [9]:
# Mancano alcune istituzioni, controlliamo quali sono e le inseriamo in un file così da poterle analizzare ed integrare
'''list_institutions = []
for inst, infos in enriched_institutions.items():
    list_institutions.append(inst)
    
contatore = []
for istituzione, cose in balance_data.items():
        if istituzione not in list_institutions:
              contatore.append(istituzione)'''

'list_institutions = []\nfor inst, infos in enriched_institutions.items():\n    list_institutions.append(inst)\n\ncontatore = []\nfor istituzione, cose in balance_data.items():\n        if istituzione not in list_institutions:\n              contatore.append(istituzione)'

In [10]:
# Nuovo dizionario per le istituzioni senza informazioni
'''istituzioni_senza_id = {}
for i, info in balance_data.items():
    if i in contatore:
        istituzioni_senza_id[i] = balance_data[i]'''

'istituzioni_senza_id = {}\nfor i, info in balance_data.items():\n    if i in contatore:\n        istituzioni_senza_id[i] = balance_data[i]'

In [11]:
# Che salviamo per integrazione nel file recupero_id_8_istituzioni
#with open('istituzioni_senza_id.json', 'w') as outfile:
#    json.dump(istituzioni_senza_id, outfile)

### Balance Data DF

In [12]:
# Rimuoviamo l'id da balance data che non ci serve più e complica le cose
for institution in balance_data:
    if 'id' in balance_data[institution]:
        del balance_data[institution]['id']

In [13]:
df = pd.DataFrame(list(balance_data.keys()), columns=['nome_istituzione'])

In [14]:
def trial(dataframe, dictionary):
    new_columns = {}
    
    for institution, information in dictionary.items():
        for year, works in information.items():
            for key, value in works.items():
                nome_colonna= f'{key}_{year}'

                if nome_colonna not in new_columns:
                    new_columns[nome_colonna] = {}
                new_columns[nome_colonna][institution] = value


    for nome_colonna, valori in new_columns.items():
        dataframe[nome_colonna] = dataframe['nome_istituzione'].map(valori)

    return dataframe

In [15]:
balance_dataframe = trial(df, balance_data)

### Enriched institutions DF

In [16]:
len(enriched_institutions_final)

1287

In [17]:
nomi = []
for id, infos in enriched_institutions_final.items():
    nomi.append(id)
    
df_istituzioni = pd.DataFrame(nomi, columns=['nome_istituzione'])

In [18]:
enriched_institutions_final

{'Institute of Neurological Sciences': {'2020': {'tot_dipendenti_per_anno': 50,
   'studiosi_usciti': 27,
   'studiosi_entrati': 21},
  '2021': {'tot_dipendenti_per_anno': 32,
   'studiosi_usciti': 15,
   'studiosi_entrati': 9},
  '2022': {'tot_dipendenti_per_anno': 24,
   'studiosi_usciti': 11,
   'studiosi_entrati': 7},
  '2023': {'tot_dipendenti_per_anno': 26,
   'studiosi_usciti': 11,
   'studiosi_entrati': 13},
  '2024': {'tot_dipendenti_per_anno': 22,
   'studiosi_usciti': 9,
   'studiosi_entrati': 7},
  'id': 'https://openalex.org/I4210120704',
  'informazioni_geografiche': {'city': 'Mangone',
   'geonames_city_id': '2524300',
   'region': None,
   'country_code': 'IT',
   'country': 'Italy',
   'latitude': 39.21430969238281,
   'longitude': 16.304439544677734,
   'regione': 'Calabria',
   'macroarea': 'Sud'},
  'counts_by_year': [{'year': 2026,
    'works_count': 10,
    'oa_works_count': 9,
    'cited_by_count': 0},
   {'year': 2025,
    'works_count': 113,
    'oa_works_count

In [19]:
def df_institutions_data(dataframe, dictionary):
    # Per ogni istituzione, estrai i dati e aggiungili come riga
    rows_data = []
    
    for institution_name, information in dictionary.items():
        row = {'nome_istituzione': institution_name}
        
        # Aggiungi le informazioni geografiche
        for key, value in information['informazioni_geografiche'].items():
            row[key] = value

        # aggiungere se è un super system
        row['is_super_system'] = information['is_super_system']

        # si aggiunge il tipo
        row['type'] = information['type']
        
        # aggiungere le info di counts_by_year
        for infos in information['counts_by_year']:
            anno = infos['year']
            for k, v in infos.items():
                if k != 'year':
                    row[f'{k}_{anno}'] = v

        # aggiungere ruoli
        ruoli = []
        for r in information['roles']:
            for k, v in r.items():
                if k == 'role':
                    ruoli.append(v)
        row['roles'] = ruoli

        # si aggiungono le summary stats
        for chiave, valore in information['summary_stats'].items():
            row[chiave] = valore
        
        rows_data.append(row)
    
    # Crea un dataframe temporaneo con i nuovi dati
    temp_df = pd.DataFrame(rows_data)
    
    # Se vuoi aggiungere questi dati a un dataframe esistente basandoti su 'institution_id'
    if not dataframe.empty and 'institution_id' in dataframe.columns:
        dataframe = dataframe.merge(temp_df, on='institution_id', how='left')
    else:
        dataframe = temp_df
    
    return dataframe

In [20]:
new_df = df_institutions_data(df_istituzioni, enriched_institutions_final)

In [21]:
new_df['roles'].value_counts()

roles
[institution]                       1092
[funder, institution]                156
[funder, institution, publisher]      32
[institution, publisher]               7
Name: count, dtype: int64

In [22]:
new_df.shape

(1287, 85)

In [23]:
pd.set_option('display.max_columns', None)
display(new_df)

,nome_istituzione,city,geonames_city_id,region,country_code,country,latitude,longitude,regione,macroarea,is_super_system,type,works_count_2026,oa_works_count_2026,cited_by_count_2026,works_count_2025,oa_works_count_2025,cited_by_count_2025,works_count_2024,oa_works_count_2024,cited_by_count_2024,works_count_2023,oa_works_count_2023,cited_by_count_2023,works_count_2022,oa_works_count_2022,cited_by_count_2022,works_count_2021,oa_works_count_2021,cited_by_count_2021,works_count_2020,oa_works_count_2020,cited_by_count_2020,works_count_2019,oa_works_count_2019,cited_by_count_2019,works_count_2018,oa_works_count_2018,cited_by_count_2018,works_count_2017,oa_works_count_2017,cited_by_count_2017,works_count_2016,oa_works_count_2016,cited_by_count_2016,works_count_2015,oa_works_count_2015,cited_by_count_2015,works_count_2014,oa_works_count_2014,cited_by_count_2014,works_count_2013,oa_works_count_2013,cited_by_count_2013,works_count_2012,oa_works_count_2012,cited_by_count_2012,works_count_2011,oa_works_count_2011,cited_by_count_2011,works_count_2010,oa_works_count_2010,cited_by_count_2010,roles,2yr_mean_citedness,h_index,i10_index,works_count_2030,oa_works_count_2030,cited_by_count_2030,works_count_9999,oa_works_count_9999,cited_by_count_9999,works_count_2028,oa_works_count_2028,cited_by_count_2028,works_count_2999,oa_works_count_2999,cited_by_count_2999,works_count_2027,oa_works_count_2027,cited_by_count_2027,works_count_2029,oa_works_count_2029,cited_by_count_2029
0,Institute of Neurological Sciences,Mangone,2524300,None,IT,Italy,39.214310,16.304440,Calabria,Sud,False,facility,10.0,9.0,0.0,113.0,71.0,118.0,108.0,72.0,491.0,117.0,82.0,960.0,94.0,72.0,2851.0,136.0,84.0,2447.0,152.0,91.0,6189.0,148.0,79.0,9442.0,230.0,135.0,13156.0,259.0,136.0,10394.0,208.0,110.0,9552.0,189.0,95.0,7873.0,173.0,90.0,15235.0,139.0,67.0,5604.0,110.0,37.0,3622.0,100.0,38.0,5466.0,78.0,24.0,3160.0,[institution],2.434053,246,8157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,University of Siena,Siena,3166548,None,IT,Italy,43.318218,11.330640,Tuscany,Centro,False,education,77.0,46.0,2.0,4554.0,3753.0,2888.0,3182.0,2542.0,13814.0,3034.0,2476.0,21889.0,2844.0,2167.0,40025.0,2662.0,1899.0,53416.0,2719.0,1808.0,75996.0,2367.0,1405.0,62857.0,2061.0,1172.0,58468.0,1987.0,1087.0,83147.0,2010.0,1064.0,91335.0,1889.0,897.0,96171.0,1882.0,860.0,74522.0,1733.0,754.0,52251.0,1870.0,810.0,67156.0,1802.0,701.0,75230.0,1711.0,618.0,61059.0,"[funder, institution]",2.391423,521,79249,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IRCCS Ospedale San Raffaele,Milan,3173435,None,IT,Italy,45.464272,9.189510,Lombardy,Nord,False,healthcare,33.0,24.0,0.0,1952.0,1290.0,2552.0,1625.0,1193.0,8190.0,1629.0,1201.0,16301.0,1426.0,987.0,34961.0,1270.0,810.0,38226.0,1163.0,689.0,49243.0,823.0,408.0,33937.0,678.0,336.0,29490.0,622.0,285.0,31595.0,682.0,243.0,33349.0,584.0,216.0,22508.0,509.0,176.0,25356.0,428.0,192.0,21717.0,429.0,155.0,22695.0,344.0,120.0,14575.0,377.0,106.0,19983.0,[institution],2.573100,375,21726,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Bambino Gesù Children's Hospital,Rome,3169070,None,IT,Italy,41.891930,12.511330,Lazio,Centro,False,healthcare,51.0,34.0,0.0,1644.0,1235.0,1750.0,1851.0,1477.0,8472.0,1627.0,1277.0,14612.0,1804.0,1374.0,31688.0,1686.0,1190.0,39876.0,1542.0,1070.0,55282.0,1138.0,644.0,36549.0,979.0,524.0,41020.0,941.0,510.0,34616.0,861.0,397.0,30021.0,791.0,394.0,32534.0,726.0,317.0,25956.0,639.0,305.0,25096.0,579.0,233.0,24339.0,532.0,207.0,21757.0,420.0,155.0,20894.0,[institution],3.008373,344,32288,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,University of Florence,Florence,3176959,None,IT,Italy,43.779251,11.246260,Tuscany,Centro,False,education,168.0,106.0,16.0,9639.0,6581.0,6729.0,8270.0,6504.0,32069.0,7698.0,6126.0,66448.0,7750.0,5796.0,100752.0,7537.0,5607.0,131602.0,7306.0,5117.0,173527.0,6296.0,3811.0,157989.0,5402.0,3232.0,152826.0,4991

### Macroaree by institution DF

Per prima cosa, trasformiamo i numeri assoluti delle produzioni di paper in percentuali, poi creiamo il dataset.

In [24]:
def papers_percentages(dictionary):
    new_dict = {}
    for id, infos in dictionary.items():
        anni = infos.get("years", {})

        for anno, valori in anni.items():
            hash_map = {}
            percentages_dict = {}
            conteggio = 0


            for identificativo, papers in valori.items():
                conteggio += papers['count']
                hash_map[papers['macroarea']] = papers['count']

            for key, value in hash_map.items():
                percentage = round((value / conteggio) * 100, 2)
                
                percentages_dict[f'{anno}_{key}'] = percentage

            if id not in new_dict:
                new_dict[id] = {
                    "nome_istituzione": infos["institution_name"]}

            new_dict[id].update(percentages_dict)            
            
    return new_dict



In [25]:
new_dict = papers_percentages(macroaree_by_institution)
new_dict

{'https://openalex.org/I4210120704': {'nome_istituzione': 'Institute of Neurological Sciences',
  '2020_Health Sciences': 74.81,
  '2020_Life Sciences': 22.22,
  '2020_Social Sciences': 1.48,
  '2020_Physical Sciences': 1.48,
  '2021_Health Sciences': 73.04,
  '2021_Life Sciences': 21.74,
  '2021_Social Sciences': 4.35,
  '2021_Physical Sciences': 0.87,
  '2022_Health Sciences': 70.37,
  '2022_Life Sciences': 19.75,
  '2022_Social Sciences': 8.64,
  '2022_Physical Sciences': 1.23,
  '2023_Health Sciences': 63.27,
  '2023_Life Sciences': 29.59,
  '2023_Social Sciences': 4.08,
  '2023_Physical Sciences': 3.06,
  '2024_Health Sciences': 65.71,
  '2024_Life Sciences': 23.81,
  '2024_Social Sciences': 5.71,
  '2024_Physical Sciences': 4.76},
 'https://openalex.org/I102064193': {'nome_istituzione': 'University of Siena',
  '2020_Health Sciences': 45.11,
  '2020_Physical Sciences': 23.3,
  '2020_Life Sciences': 17.44,
  '2020_Social Sciences': 14.16,
  '2021_Health Sciences': 43.66,
  '2021_P

In [26]:
def df_papers_data(dictionary):
    rows_data = []

    for institution_id, information in dictionary.items():
        row = {}

        for key, value in information.items():
            row[key] = value

        rows_data.append(row)

    dataframe = pd.DataFrame(rows_data)
    return dataframe


In [27]:
macroaree_df = df_papers_data(new_dict)

In [28]:
macroaree_df

,nome_istituzione,2020_Health Sciences,2020_Life Sciences,2020_Social Sciences,2020_Physical Sciences,2021_Health Sciences,2021_Life Sciences,2021_Social Sciences,2021_Physical Sciences,2022_Health Sciences,2022_Life Sciences,2022_Social Sciences,2022_Physical Sciences,2023_Health Sciences,2023_Life Sciences,2023_Social Sciences,2023_Physical Sciences,2024_Health Sciences,2024_Life Sciences,2024_Social Sciences,2024_Physical Sciences
0,Institute of Neurological Sciences,74.81,22.22,1.48,1.48,73.04,21.74,4.35,0.87,70.37,19.75,8.64,1.23,63.27,29.59,4.08,3.06,65.71,23.81,5.71,4.76
1,University of Siena,45.11,17.44,14.16,23.30,43.66,17.15,16.17,23.02,44.97,17.19,13.95,23.89,42.75,16.65,15.60,25.01,45.32,15.95,15.75,22.97
2,IRCCS Ospedale San Raffaele,82.78,13.71,2.00,1.50,79.04,17.56,1.42,1.98,73.76,21.86,2.36,2.03,75.72,19.89,2.48,1.92,72.41,23.45,1.57,2.57
3,Bambino Gesù Children's Hospital,66.37,28.78,2.15,2.70,65.48,28.03,4.12,2.37,66.84,27.35,3.90,1.92,69.99,25.60,2.28,2.13,69.21,25.67,2.96,2.16
4,University of Florence,32.44,17.50,15.44,34.62,31.94,17.75,15.81,34.51,32.98,16.92,16.39,33.72,33.15,15.82,16.33,34.70,31.83,14.80,17.52,35.85
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,Ospedale Pediatrico Giovanni XXIII,75.86,18.97,2.59,2.59,80.33,13.93,3.28,2.46,74.36,21.79,2.56,1.28,77.54,19.57,1.45,1.45,77.54,16.67,2.90,2.90
1283,Ospedale San Bortolo,85.17,10.27,1.90,2.66,88.55,7.25,1.53,2.67,87.81,10.04,1.08,1.08,83.84,10.44,1.35,4.38,84.62,9.79,2.10,3.50
1284,Telethon Foundation,50.00,37.50,12.50,NaN,40.00,50.00,NaN,10.00,10.00,80.00,NaN,10.00,57.14,42.86,NaN,NaN,28.57,71.43,NaN,NaN
1285,Tecnologia e Ricerca Italiana (Italy),NaN,NaN,NaN,100.00,NaN,9.09,NaN,90.91,NaN,16.67,NaN,83.33,NaN,33.33,NaN,66.67,NaN,20.00,NaN,80.00


## Merge dei due df

In [29]:
data = new_df.merge(balance_dataframe, on= ['nome_istituzione'], how='inner')

In [30]:
final_data = data.merge(macroaree_df, on = ['nome_istituzione'], how='inner')

In [31]:
final_data

,nome_istituzione,city,geonames_city_id,region,country_code,country,latitude,longitude,regione,macroarea,is_super_system,type,works_count_2026,oa_works_count_2026,cited_by_count_2026,works_count_2025,oa_works_count_2025,cited_by_count_2025,works_count_2024,oa_works_count_2024,cited_by_count_2024,works_count_2023,oa_works_count_2023,cited_by_count_2023,works_count_2022,oa_works_count_2022,cited_by_count_2022,works_count_2021,oa_works_count_2021,cited_by_count_2021,works_count_2020,oa_works_count_2020,cited_by_count_2020,works_count_2019,oa_works_count_2019,cited_by_count_2019,works_count_2018,oa_works_count_2018,cited_by_count_2018,works_count_2017,oa_works_count_2017,cited_by_count_2017,works_count_2016,oa_works_count_2016,cited_by_count_2016,works_count_2015,oa_works_count_2015,cited_by_count_2015,works_count_2014,oa_works_count_2014,cited_by_count_2014,works_count_2013,oa_works_count_2013,cited_by_count_2013,works_count_2012,oa_works_count_2012,cited_by_count_2012,works_count_2011,oa_works_count_2011,cited_by_count_2011,works_count_2010,oa_works_count_2010,cited_by_count_2010,roles,2yr_mean_citedness,h_index,i10_index,works_count_2030,oa_works_count_2030,cited_by_count_2030,works_count_9999,oa_works_count_9999,cited_by_count_9999,works_count_2028,oa_works_count_2028,cited_by_count_2028,works_count_2999,oa_works_count_2999,cited_by_count_2999,works_count_2027,oa_works_count_2027,cited_by_count_2027,works_count_2029,oa_works_count_2029,cited_by_count_2029,tot_dipendenti_per_anno_2019,studiosi_usciti_2019,studiosi_entrati_2019,tot_dipendenti_per_anno_2020,studiosi_usciti_2020,studiosi_entrati_2020,tot_dipendenti_per_anno_2021,studiosi_usciti_2021,studiosi_entrati_2021,tot_dipendenti_per_anno_2022,studiosi_usciti_2022,studiosi_entrati_2022,tot_dipendenti_per_anno_2023,studiosi_usciti_2023,studiosi_entrati_2023,tot_dipendenti_per_anno_2024,studiosi_usciti_2024,studiosi_entrati_2024,2020_Health Sciences,2020_Life Sciences,2020_Social Sciences,2020_Physical Sciences,2021_Health Sciences,2021_Life Sciences,2021_Social Sciences,2021_Physical Sciences,2022_Health Sciences,2022_Life Sciences,2022_Social Sciences,2022_Physical Sciences,2023_Health Sciences,2023_Life Sciences,2023_Social Sciences,2023_Physical Sciences,2024_Health Sciences,2024_Life Sciences,2024_Social Sciences,2024_Physical Sciences
0,Institute of Neurological Sciences,Mangone,2524300,None,IT,Italy,39.214310,16.304440,Calabria,Sud,False,facility,10.0,9.0,0.0,113.0,71.0,118.0,108.0,72.0,491.0,117.0,82.0,960.0,94.0,72.0,2851.0,136.0,84.0,2447.0,152.0,91.0,6189.0,148.0,79.0,9442.0,230.0,135.0,13156.0,259.0,136.0,10394.0,208.0,110.0,9552.0,189.0,95.0,7873.0,173.0,90.0,15235.0,139.0,67.0,5604.0,110.0,37.0,3622.0,100.0,38.0,5466.0,78.0,24.0,3160.0,[institution],2.434053,246,8157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.0,17.0,0.0,50.0,35.0,29.0,28.0,20.0,13.0,22.0,12.0,14.0,24.0,11.0,14.0,20.0,10.0,7.0,74.81,22.22,1.48,1.48,73.04,21.74,4.35,0.87,70.37,19.75,8.64,1.23,63.27,29.59,4.08,3.06,65.71,23.81,5.71,4.76
1,University of Siena,Siena,3166548,None,IT,Italy,43.318218,11.330640,Tuscany,Centro,False,education,77.0,46.0,2.0,4554.0,3753.0,2888.0,3182.0,2542.0,13814.0,3034.0,2476.0,21889.0,2844.0,2167.0,40025.0,2662.0,1899.0,53416.0,2719.0,1808.0,75996.0,2367.0,1405.0,62857.0,2061.0,1172.0,58468.0,1987.0,1087.0,83147.0,2010.0,1064.0,91335.0,1889.0,897.0,96171.0,1882.0,860.0,74522.0,1733.0,754.0,52251.0,1870.0,810.0,67156.0,1802.0,701.0,75230.0,1711.0,618.0,61059.0,"[funder, institution]",2.391423,521,79249,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,415.0,73.0,0.0,394.0,63.0,52.0,400.0,77.0,69.0,398.0,102.0,75.0,438.0,151.0,142.0,349.0,93.0,62.0,45.11,17.44,14.16,23.30,43.66,17.15,16.17,23.02,44.97,17.19,13.95,23.89,42.75,16.65,15.60,25.01,45.32,15.95,15.75,22.97
2,IRCCS Ospedale San Raffaele,Milan,3173435,None,IT,Italy,45.464272,9.189510,Lombardy,Nord,False,healthcare,33.0,24.0,0.0,1952.0,1290.0,2552.0,1625.0,1193.0

In [33]:
final_data.to_csv('data.csv', index=False)